# Nível 1 — Triagem PLD: dados e primeira análise com LLM

**Mensagem arquitetural do projeto inteiro:** Python calcula. O LLM interpreta.

Todo número que aparece neste notebook (soma, mediana, contagem, comparação com limite,
conversão de moeda) é produzido por pandas. A LLM só entra depois, para ler os fatos já
calculados e escrever uma interpretação qualitativa — nunca para fazer contas.

Dados fictícios gerados para fins de avaliação, sem relação com clientes ou operações reais.

## Parte A — Tratamento e regras (pandas)

### 1. Carregamento dos dados

In [1]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

CAMINHO_DADOS = Path("../dados/dados_nivel_1.json")

with open(CAMINHO_DADOS, encoding="utf-8") as f:
    raw = json.load(f)

taxa_cambio_usd_brl = raw["taxa_cambio_usd_brl"]
df = pd.DataFrame(raw["operacoes"])

print(f"taxa_cambio_usd_brl = {taxa_cambio_usd_brl}")
print(f"linhas carregadas: {len(df)}")
df.head()

taxa_cambio_usd_brl = 5.4
linhas carregadas: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB


### 2. Inspeção inicial e identificação dos problemas de qualidade

In [3]:
print("IDs únicos:", df["id"].nunique(), "de", len(df), "linhas")
contagem_ids = df["id"].value_counts()
ids_duplicados = contagem_ids[contagem_ids > 1]
print("\nIDs que aparecem mais de uma vez:")
print(ids_duplicados)

IDs únicos: 19 de 20 linhas

IDs que aparecem mais de uma vez:
id
OP-0007    2
Name: count, dtype: int64


In [4]:
# As linhas com ID duplicado são idênticas em todos os campos, ou o conteúdo diverge?
df[df["id"].isin(ids_duplicados.index)].sort_values("id")

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [5]:
print("Datas nulas:", df["data"].isna().sum())
df[df["data"].isna()]

Datas nulas: 1


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


In [6]:
print("Moedas presentes:", df["moeda"].value_counts().to_dict())
df[df["moeda"] != "BRL"]

Moedas presentes: {'BRL': 19, 'USD': 1}


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional


In [7]:
print("Canais:", df["canal"].value_counts().to_dict())
print("Tipos:", df["tipo"].value_counts().to_dict())
print("Valores <= 0:", (df["valor"] <= 0).sum())
print("Valores nulos:", df["valor"].isna().sum())
df["valor"].describe()

Canais: {'pix': 9, 'ted': 5, 'boleto': 3, 'cartao': 2, 'especie': 1}
Tipos: {'transferencia_enviada': 11, 'pagamento': 5, 'transferencia_recebida': 3, 'deposito': 1}
Valores <= 0: 0
Valores nulos: 0


count       20.000000
mean     11495.000000
std       7983.369227
min       1400.000000
25%       4175.000000
50%      10400.000000
75%      17225.000000
max      27000.000000
Name: valor, dtype: float64

### O que encontramos e como tratamos

**1. Duplicata exata de linha — `OP-0007` (CLI-A-3, 2026-03-05, R$ 17.200).**
A linha aparece duas vezes com *todos* os campos idênticos — não é coincidência de
negócio (duas operações diferentes de mesmo valor no mesmo dia teriam IDs distintos),
é reprocessamento do sistema legado. Tratamento: `drop_duplicates()` na linha inteira,
mantendo a primeira ocorrência. **Impacto se não tratássemos:** o grupo
CLI-A-3/2026-03-05 passaria de 3 para 4 operações somando R$ 65.700 (>R$ 50.000) e a
Regra 1 dispararia incorretamente para esse cliente — um falso positivo só por causa
da duplicata. Isso é verificado explicitamente na célula de validação da Regra 1 mais
abaixo.

**2. Data ausente — `OP-0017` (CLI-A-5), `data: null`, observação "data nao capturada
pelo sistema".** Não temos como inferir a data real sem inventar informação. Tratamento:
convertemos para `NaT` (não removemos a linha) e criamos a flag `data_ausente` para que
fique auditável. **Impacto:** a Regra 1 depende de "mesma data", então essa operação é
excluída *daquele agrupamento* — mas continua contando no volume total do cliente e é
elegível para a Regra 2 (valor atípico), que não depende de data.

**3. Operação em moeda estrangeira — `OP-0013` (CLI-A-4), USD, "remessa internacional".**
Não é um erro de qualidade, é um dado legítimo que precisa de conversão antes de entrar
em qualquer regra baseada em valor. Tratamento: `valor_brl = valor * taxa_cambio_usd_brl`
quando `moeda == "USD"`, mantendo `valor` e `moeda` originais intactos para auditoria.

Não há valores nulos ou ≤ 0 em `valor`, e canais/tipos têm vocabulário controlado e
consistente — nenhum tratamento adicional necessário aí.

### 3. Limpeza: remover duplicata, tratar data, normalizar valores para BRL

In [8]:
df_limpo = df.drop_duplicates().copy()
print(f"Linhas após remover duplicata exata: {len(df_limpo)} (era {len(df)})")

df_limpo["data"] = pd.to_datetime(df_limpo["data"], errors="coerce")
df_limpo["data_ausente"] = df_limpo["data"].isna()

assert set(df_limpo["moeda"].unique()) <= {"BRL", "USD"}, "moeda inesperada encontrada"
df_limpo["valor_brl"] = df_limpo["valor"].where(
    df_limpo["moeda"] == "BRL", df_limpo["valor"] * taxa_cambio_usd_brl
)

df_limpo[["id", "cliente_id", "data", "data_ausente", "valor", "moeda", "valor_brl"]]

Linhas após remover duplicata exata: 19 (era 20)


,id,cliente_id,data,data_ausente,valor,moeda,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,False,18100,BRL,18100.0
1,OP-0002,CLI-A-1,2026-03-09,False,17300,BRL,17300.0
2,OP-0003,CLI-A-1,2026-03-09,False,18800,BRL,18800.0
3,OP-0004,CLI-A-1,2026-03-21,False,3300,BRL,3300.0
4,OP-0005,CLI-A-2,2026-03-14,False,25900,BRL,25900.0
5,OP-0006,CLI-A-2,2026-03-14,False,27000,BRL,27000.0
6,OP-0007,CLI-A-3,2026-03-05,False,17200,BRL,17200.0
7,OP-0008,CLI-A-3,2026-03-05,False,15200,BRL,15200.0
8,OP-0009,CLI-A-3,2026-03-05,False,16100,BRL,16100.0
10,OP-0010,CLI-A-4,2026-03-03,False,3800,BRL,3800.0


### 4. Agregações (pandas puro, sem LLM)

In [9]:
volume_por_cliente = (
    df_limpo.groupby("cliente_id")["valor_brl"]
    .sum()
    .rename("volume_total_brl")
    .sort_values(ascending=False)
    .reset_index()
)
volume_por_cliente

,cliente_id,volume_total_brl
0,CLI-A-4,79500.0
1,CLI-A-1,57500.0
2,CLI-A-2,52900.0
3,CLI-A-3,48500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


In [10]:
operacoes_por_canal = (
    df_limpo["canal"].value_counts().rename_axis("canal").reset_index(name="qtd_operacoes")
)
operacoes_por_canal

,canal,qtd_operacoes
0,pix,8
1,ted,5
2,boleto,3
3,cartao,2
4,especie,1


### 5. Regra 1 — Fracionamento

Sinaliza o **cliente** quando, numa **mesma data**: 3+ operações, soma > R$ 50.000,
e nenhuma operação isolada atinge R$ 20.000. Operações sem data são excluídas do
agrupamento (não dá para agrupar por uma data que não existe), mas continuam na base.

In [11]:
LIMITE_SOMA = 50_000.0
LIMITE_OP_ISOLADA = 20_000.0
MIN_OPERACOES = 3

elegiveis_regra1 = df_limpo[~df_limpo["data_ausente"]]
agrupado = elegiveis_regra1.groupby(["cliente_id", "data"])["valor_brl"].agg(
    qtd="count", soma="sum", maximo="max"
)
agrupado["regra_fracionamento"] = (
    (agrupado["qtd"] >= MIN_OPERACOES)
    & (agrupado["soma"] > LIMITE_SOMA)
    & (agrupado["maximo"] < LIMITE_OP_ISOLADA)
)
agrupado

qtd     soma   maximo  regra_fracionamento
cliente_id data                                                  
CLI-A-1    2026-03-09    3  54200.0  18800.0                 True
           2026-03-21    1   3300.0   3300.0                False
CLI-A-2    2026-03-14    2  52900.0  27000.0                False
CLI-A-3    2026-03-05    3  48500.0  17200.0                False
CLI-A-4    2026-03-03    1   3800.0   3800.0                False
           2026-03-11    1   5100.0   5100.0                False
           2026-03-18    1   5800.0   5800.0                False
           2026-03-24    1  64800.0  64800.0                False
CLI-A-5    2026-03-07    1   2900.0   2900.0                False
           2026-03-16    1   7000.0   7000.0                False
           2026-03-26    1   2700.0   2700.0                False
CLI-A-6    2026-03-12    1   8800.0   8800.0                False
           2026-03-28    1   1400.0   1400.0                False

In [12]:
df_limpo["regra_fracionamento"] = False
grupos_flagrados = agrupado[agrupado["regra_fracionamento"]].index
if len(grupos_flagrados) > 0:
    chave = elegiveis_regra1.set_index(["cliente_id", "data"]).index.isin(grupos_flagrados)
    idx_flagrado = elegiveis_regra1.index[chave]
    df_limpo.loc[idx_flagrado, "regra_fracionamento"] = True

df_limpo[df_limpo["regra_fracionamento"]][
    ["id", "cliente_id", "data", "valor_brl", "regra_fracionamento"]
]

,id,cliente_id,data,valor_brl,regra_fracionamento
0,OP-0001,CLI-A-1,2026-03-09,18100.0,True
1,OP-0002,CLI-A-1,2026-03-09,17300.0,True
2,OP-0003,CLI-A-1,2026-03-09,18800.0,True


### 6. Validação da Regra 1 — caso capturado vs. caso parecido que não deveria disparar

In [13]:
print("CASO POSITIVO — deveria disparar: CLI-A-1 em 2026-03-09")
print(agrupado.loc[("CLI-A-1", "2026-03-09")])
print("-> 3 operações, soma R$54.200 (>50k), máxima R$18.800 (<20k) => dispara. Confirmado acima.\n")

print("CASO PARECIDO — não deveria disparar: CLI-A-3 em 2026-03-05")
print(agrupado.loc[("CLI-A-3", "2026-03-05")])
print(
    "-> também são 3 operações na mesma data (após remover a duplicata do OP-0007), "
    "mas a soma é R$48.500, abaixo do limite de R$50.000 => não dispara.\n"
)

print("Por que este par importa: sem a deduplicação do passo 3, CLI-A-3 teria 4 "
      "operações somando R$65.700 nessa data e a Regra 1 dispararia incorretamente. "
      "A limpeza de dados não é cosmética — ela decide o resultado da regra.")

assert bool(agrupado.loc[("CLI-A-1", "2026-03-09"), "regra_fracionamento"]) is True
assert bool(agrupado.loc[("CLI-A-3", "2026-03-05"), "regra_fracionamento"]) is False
print("\nOK: validação passou (asserts).")

CASO POSITIVO — deveria disparar: CLI-A-1 em 2026-03-09
qtd                          3
soma                   54200.0
maximo                 18800.0
regra_fracionamento       True
Name: (CLI-A-1, 2026-03-09 00:00:00), dtype: object
-> 3 operações, soma R$54.200 (>50k), máxima R$18.800 (<20k) => dispara. Confirmado acima.

CASO PARECIDO — não deveria disparar: CLI-A-3 em 2026-03-05
qtd                          3
soma                   48500.0
maximo                 17200.0
regra_fracionamento      False
Name: (CLI-A-3, 2026-03-05 00:00:00), dtype: object
-> também são 3 operações na mesma data (após remover a duplicata do OP-0007), mas a soma é R$48.500, abaixo do limite de R$50.000 => não dispara.

Por que este par importa: sem a deduplicação do passo 3, CLI-A-3 teria 4 operações somando R$65.700 nessa data e a Regra 1 dispararia incorretamente. A limpeza de dados não é cosmética — ela decide o resultado da regra.

OK: validação passou (asserts).


### 7. Regra 2 — Valor atípico

Sinaliza a **operação** cujo `valor_brl` é maior que 5x a mediana dos valores daquele
cliente, aplicada somente a clientes com 4+ operações.

In [14]:
MULTIPLICADOR = 5
MIN_OPERACOES_ATIPICO = 4

mediana_por_cliente = df_limpo.groupby("cliente_id")["valor_brl"].transform("median")
qtd_por_cliente = df_limpo.groupby("cliente_id")["valor_brl"].transform("count")

df_limpo["mediana_cliente_brl"] = mediana_por_cliente
df_limpo["regra_valor_atipico"] = (qtd_por_cliente >= MIN_OPERACOES_ATIPICO) & (
    df_limpo["valor_brl"] > MULTIPLICADOR * mediana_por_cliente
)

df_limpo[
    ["id", "cliente_id", "valor", "moeda", "valor_brl", "mediana_cliente_brl", "regra_valor_atipico"]
][qtd_por_cliente >= MIN_OPERACOES_ATIPICO]

,id,cliente_id,valor,moeda,valor_brl,mediana_cliente_brl,regra_valor_atipico
0,OP-0001,CLI-A-1,18100,BRL,18100.0,17700.0,False
1,OP-0002,CLI-A-1,17300,BRL,17300.0,17700.0,False
2,OP-0003,CLI-A-1,18800,BRL,18800.0,17700.0,False
3,OP-0004,CLI-A-1,3300,BRL,3300.0,17700.0,False
10,OP-0010,CLI-A-4,3800,BRL,3800.0,5450.0,False
11,OP-0011,CLI-A-4,5100,BRL,5100.0,5450.0,False
12,OP-0012,CLI-A-4,5800,BRL,5800.0,5450.0,False
13,OP-0013,CLI-A-4,12000,USD,64800.0,5450.0,True
14,OP-0014,CLI-A-5,2900,BRL,2900.0,3600.0,False
15,OP-0015,CLI-A-5,7000,BRL,7000.0,3600.0,False


In [15]:
flagrados_regra2 = df_limpo[df_limpo["regra_valor_atipico"]]
print(f"{len(flagrados_regra2)} operação(ões) sinalizada(s) pela Regra 2:")
flagrados_regra2[["id", "cliente_id", "valor", "moeda", "valor_brl", "mediana_cliente_brl"]]

1 operação(ões) sinalizada(s) pela Regra 2:


,id,cliente_id,valor,moeda,valor_brl,mediana_cliente_brl
13,OP-0013,CLI-A-4,12000,USD,64800.0,5450.0


A única operação sinalizada é `OP-0013` (CLI-A-4): R$ 12.000 USD convertidos para
R$ 64.800, contra uma mediana de R$ 5.450 do cliente (limite = 5×5.450 = R$ 27.250).
É o caso plantado que testa conversão de moeda **e** detecção de outlier ao mesmo tempo —
se a conversão para BRL não tivesse sido feita antes de comparar com a mediana, essa
operação pareceria um valor comum (R$ 12.000) e a regra não dispararia.

### Conferência: nivel_2/tools.py reproduz o mesmo resultado?

O Nível 2 reaproveita esta mesma lógica de limpeza e regras, mas fatorada em
`nivel_2/tools.py` (carga, limpeza, regras e ferramentas do agente, tudo num módulo só
-- ver docs/DECISOES.md para o porquê da consolidação) para rodar sobre a base maior
sem duplicar código. Este notebook mantém a versão narrada, célula a célula, para
deixar o raciocínio explícito -- a célula abaixo prova que as duas versões concordam
neste dataset, então não há lógica divergente entre nível 1 e nível 2.

In [16]:
sys.path.insert(0, "..")
from nivel_2.tools import carregar_e_limpar, aplicar_regras

df_via_modulo = aplicar_regras(carregar_e_limpar(CAMINHO_DADOS))

comparaveis = ["id", "cliente_id", "regra_fracionamento", "regra_valor_atipico"]
a = df_limpo[comparaveis].sort_values("id").reset_index(drop=True)
b = df_via_modulo[comparaveis].sort_values("id").reset_index(drop=True)
pd.testing.assert_frame_equal(a, b)
print("OK: nivel_2/tools.py produz exatamente o mesmo resultado.")

OK: nivel_2/tools.py produz exatamente o mesmo resultado.


## Parte B — Análise com LLM

### 7. Escolha do cliente e montagem do contexto (fatos, não cálculos)

Escolhemos **CLI-A-1**, sinalizado pela Regra 1 (fracionamento em 2026-03-09). O
dict abaixo é montado **inteiramente com pandas** — é o único material que vai para a
LLM. Note que não enviamos "calcule a soma" ou "esse valor passou de X?" — enviamos os
números já prontos e pedimos interpretação.

In [17]:
cliente_escolhido = "CLI-A-1"
sub_cliente = df_limpo[df_limpo["cliente_id"] == cliente_escolhido]

contexto_cliente = {
    "cliente_id": cliente_escolhido,
    "quantidade_operacoes": int(len(sub_cliente)),
    "volume_total_brl": round(float(sub_cliente["valor_brl"].sum()), 2),
    "regra_fracionamento": bool(sub_cliente["regra_fracionamento"].any()),
    "regra_valor_atipico": bool(sub_cliente["regra_valor_atipico"].any()),
    "operacoes_relevantes": [
        {
            "id": row["id"],
            "data": row["data"].date().isoformat() if not row["data_ausente"] else None,
            "valor_brl": round(float(row["valor_brl"]), 2),
            "canal": row["canal"],
            "tipo": row["tipo"],
            "contraparte": row["contraparte"],
        }
        for _, row in sub_cliente.iterrows()
    ],
}
contexto_cliente

{'cliente_id': 'CLI-A-1',
 'quantidade_operacoes': 4,
 'volume_total_brl': 57500.0,
 'regra_fracionamento': True,
 'regra_valor_atipico': False,
 'operacoes_relevantes': [{'id': 'OP-0001',
   'data': '2026-03-09',
   'valor_brl': 18100.0,
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA'},
  {'id': 'OP-0002',
   'data': '2026-03-09',
   'valor_brl': 17300.0,
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA'},
  {'id': 'OP-0003',
   'data': '2026-03-09',
   'valor_brl': 18800.0,
   'canal': 'ted',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Beta Servicos ME'},
  {'id': 'OP-0004',
   'data': '2026-03-21',
   'valor_brl': 3300.0,
   'canal': 'boleto',
   'tipo': 'pagamento',
   'contraparte': 'Gama Distribuidora'}]}

### 8-9. Saída estruturada validada com Pydantic + tratamento de resposta malformada

Reaproveitamos `nivel_2.agente.Parecer` (mesmo contrato usado pelo agente do Nível 2 --
uma única definição do formato de saída, não duas).

In [18]:
from pydantic import ValidationError

from nivel_2.agente import Parecer, extrair_json, get_llm_config

Parecer.model_fields

{'nivel_risco': FieldInfo(annotation=Literal['baixo', 'médio', 'alto'], required=True),
 'tipologia_suspeita': FieldInfo(annotation=str, required=True),
 'red_flags': FieldInfo(annotation=list[str], required=False, default_factory=list),
 'justificativa': FieldInfo(annotation=str, required=True)}

In [19]:
def chamar_llm_parecer(system_instruction: str, user_prompt: str, client_config):
    '''Chamada single-shot (sem tools) para o Nível 1: so pede o parecer estruturado.

    Retorna (parecer_validado_ou_None, metricas_dict). Trata JSON invalido, campos
    ausentes/invalidos, erro de API e timeout sem deixar a celula quebrar.
    '''
    inicio = time.perf_counter()
    if not client_config.configurado:
        return None, {
            "status": "erro",
            "erro": "GEMINI_API_KEY nao configurada (ver .env.example). "
                    "Preencha o .env com uma chave gratuita do Google AI Studio para executar esta celula de verdade.",
            "tempo_resposta_s": round(time.perf_counter() - inicio, 3),
            "tokens_entrada": None,
            "tokens_saida": None,
            "tokens_totais": None,
            "modelo": client_config.model,
            "provedor": client_config.provider,
        }

    try:
        from google import genai
        from google.genai import types

        client = genai.Client(api_key=client_config.api_key)
        resposta = client.models.generate_content(
            model=client_config.model,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1,
            ),
        )
        tempo = time.perf_counter() - inicio
        uso = getattr(resposta, "usage_metadata", None)
        tokens_entrada = getattr(uso, "prompt_token_count", None) if uso else None
        tokens_saida = getattr(uso, "candidates_token_count", None) if uso else None
        tokens_totais = getattr(uso, "total_token_count", None) if uso else None

        bruto = extrair_json(resposta.text)
        if bruto is None:
            return None, {
                "status": "malformado", "erro": "resposta nao continha JSON valido",
                "tempo_resposta_s": round(tempo, 3), "tokens_entrada": tokens_entrada,
                "tokens_saida": tokens_saida, "tokens_totais": tokens_totais,
                "modelo": client_config.model, "provedor": client_config.provider,
            }
        try:
            parecer = Parecer(**bruto)
        except ValidationError as exc:
            return None, {
                "status": "malformado", "erro": str(exc),
                "tempo_resposta_s": round(tempo, 3), "tokens_entrada": tokens_entrada,
                "tokens_saida": tokens_saida, "tokens_totais": tokens_totais,
                "modelo": client_config.model, "provedor": client_config.provider,
            }

        return parecer, {
            "status": "ok", "erro": None,
            "tempo_resposta_s": round(tempo, 3), "tokens_entrada": tokens_entrada,
            "tokens_saida": tokens_saida, "tokens_totais": tokens_totais,
            "modelo": client_config.model, "provedor": client_config.provider,
        }
    except Exception as exc:
        tempo = time.perf_counter() - inicio
        status = "timeout" if "timeout" in str(exc).lower() else "erro"
        return None, {
            "status": status, "erro": str(exc),
            "tempo_resposta_s": round(tempo, 3), "tokens_entrada": None,
            "tokens_saida": None, "tokens_totais": None,
            "modelo": client_config.model, "provedor": client_config.provider,
        }

### 10-11. Prompt V1 (simples) vs Prompt V2 (estruturado) — execução e comparação

In [20]:
PROMPT_V1_SYSTEM = (
    "Voce e um analista de PLD. Olhe os dados do cliente e diga se ele e suspeito, "
    "de que tipo, e por que. Responda em JSON com nivel_risco, tipologia_suspeita, "
    "red_flags e justificativa."
)

PROMPT_V2_SYSTEM = (
    "Voce e um analista de Prevencao a Lavagem de Dinheiro (PLD) em um banco.\n"
    "Os fatos abaixo ja foram CALCULADOS por um sistema deterministico em pandas "
    "(contagens, somas, flags de regra) -- voce nao deve recalcular, questionar ou "
    "inventar nenhum numero.\n"
    "Uma flag de regra e um indicio estatistico, NAO e prova de atividade ilicita: "
    "avalie o conjunto de evidencias antes de concluir.\n"
    "Distinga claramente, na justificativa, o que e FATO (o que os dados mostram) do "
    "que e HIPOTESE (sua inferencia sobre o comportamento).\n"
    "Nao invente operacoes, valores ou contrapartes que nao estejam nos fatos fornecidos.\n"
    "Responda EXCLUSIVAMENTE com um JSON valido (sem markdown, sem texto fora do JSON) "
    "no formato: {\"nivel_risco\": \"baixo|medio|alto\", \"tipologia_suspeita\": \"...\", "
    "\"red_flags\": [\"...\"], \"justificativa\": \"...\"}"
)

user_prompt_comum = "Fatos do cliente:\n" + json.dumps(contexto_cliente, ensure_ascii=False, indent=2)

config_llm = get_llm_config()
print(f"Provedor configurado: {config_llm.provider} | modelo: {config_llm.model} | "
      f"chave presente: {config_llm.configurado}")

Provedor configurado: google-ai-studio | modelo: gemini-flash-lite-latest | chave presente: True


In [21]:
parecer_v1, metricas_v1 = chamar_llm_parecer(PROMPT_V1_SYSTEM, user_prompt_comum, config_llm)
print("--- Prompt V1 ---")
print("metricas:", json.dumps(metricas_v1, ensure_ascii=False, indent=2))
print("parecer:", parecer_v1.model_dump() if parecer_v1 else None)

--- Prompt V1 ---
metricas: {
  "status": "ok",
  "erro": null,
  "tempo_resposta_s": 189.774,
  "tokens_entrada": 492,
  "tokens_saida": 313,
  "tokens_totais": 805,
  "modelo": "gemini-flash-lite-latest",
  "provedor": "google-ai-studio"
}
parecer: {'nivel_risco': 'alto', 'tipologia_suspeita': 'Fracionamento de Operações (Smurfing / Estruturação) para Ocultação de Origem/Destino de Recursos', 'red_flags': ['Ativação da regra de fracionamento de valores', 'Realização de múltiplas operações de valores elevados e próximos em um único dia (09/03/2026)', 'Concentração de transferências enviadas para contrapartes comerciais (Alfa Comercio LTDA e Beta Servicos ME) logo após a entrada ou disponibilidade de fundos', 'Uso misto de canais (PIX e TED) para movimentar quantias que individualmente parecem buscar evitar limiares tradicionais de reporte ou escrutínio'], 'justificativa': 'O cliente apresentou um padrão clássico de estruturação (fracionamento), realizando três transações de valores ex

In [22]:
parecer_v2, metricas_v2 = chamar_llm_parecer(PROMPT_V2_SYSTEM, user_prompt_comum, config_llm)
print("--- Prompt V2 ---")
print("metricas:", json.dumps(metricas_v2, ensure_ascii=False, indent=2))
print("parecer:", parecer_v2.model_dump() if parecer_v2 else None)

--- Prompt V2 ---
metricas: {
  "status": "ok",
  "erro": null,
  "tempo_resposta_s": 22.959,
  "tokens_entrada": 656,
  "tokens_saida": 275,
  "tokens_totais": 931,
  "modelo": "gemini-flash-lite-latest",
  "provedor": "google-ai-studio"
}
parecer: {'nivel_risco': 'médio', 'tipologia_suspeita': 'Fracionamento de valores (Smurfing)', 'red_flags': ['regra_fracionamento acionada', 'multiplas transferencias de valores proximos no mesmo dia'], 'justificativa': 'FATO: O sistema registrou 4 operacoes totalizando R$ 57.500,00, com a flag de regra_fracionamento ativada. No dia 2026-03-09, houve tres transferencias enviadas em curto espaco de tempo e com valores proximos (R$ 18.100,00 e R$ 17.300,00 para Alfa Comercio LTDA; R$ 18.800,00 para Beta Servicos ME), alem de um pagamento de R$ 3.300,00 via boleto em 2026-03-21 para Gama Distribuidora. HIPOTESE: O agrupamento de transferencias de valores elevados em um mesmo dia sugere uma tentativa de fracionamento para evitar controles formais de mon

### Comparação V1 vs V2

**Se `GEMINI_API_KEY` não estiver configurada nesta execução**, as duas células acima
mostram o tratamento de erro real (não uma simulação) — a função captura a ausência de
chave, registra `status: "erro"` com uma mensagem clara, e o notebook continua rodando
sem quebrar. Esse é exatamente o comportamento que `nivel_2/agente.py` e `nivel_2/lote.py`
também têm ao processar os 10 clientes do Nível 2 sem chave configurada. Para obter uma
comparação real de conteúdo, basta preencher `GEMINI_API_KEY` no `.env` (gratuito via
[Google AI Studio](https://aistudio.google.com/apikey)) e reexecutar o notebook — ver
`docs/DECISOES.md` para o status exato de cada item que depende disso.

**Diferença esperada por desenho de prompt** (a validar quando houver execução real):
- **V1** não deixa explícito que os números já são definitivos, não distingue fato de
  hipótese, e não proíbe a LLM de "recalcular" ou questionar os valores — mais sujeito a
  a LLM tratar uma flag como prova definitiva de ilicitude, ou hedgear demais por
  insegurança sobre os números.
- **V2** força a separação fato/hipótese na justificativa, proíbe explicitamente
  invenção de dados e recomputação, e é mais rígido sobre o formato de saída — deve
  produzir `red_flags` mais rastreáveis aos fatos enviados e uma `justificativa` mais
  auditável (dá para checar cada afirmação contra o `contexto_cliente`).

### 10 (cont.) Registro de tokens e latência

Consolidando as métricas das duas chamadas num único quadro, como pedido no enunciado.

In [23]:
metricas_df = pd.DataFrame([
    {"prompt": "V1", **metricas_v1},
    {"prompt": "V2", **metricas_v2},
])
metricas_df

,prompt,status,erro,tempo_resposta_s,tokens_entrada,tokens_saida,tokens_totais,modelo,provedor
0,V1,ok,None,189.774,492,313,805,gemini-flash-lite-latest,google-ai-studio
1,V2,ok,None,22.959,656,275,931,gemini-flash-lite-latest,google-ai-studio
